# 🛡️ Zero-Trust Remote LLM & VLM Inference Gateway (`v1.3.0`)

Wrap any OpenAI-compatible local inference engine (`llama-server` / `vLLM` / `Ollama` bound strictly to `127.0.0.1:8080`) behind an **application-layer AES-256-GCM AEAD gateway** on shared or untrusted cloud GPUs (RunPod, Vast.ai, Lambda, AutoDL, or SSH port-forwarded Jupyter pods).

- **Zero Edge Plaintext**: Prompts, system instructions, multimodal images, and streaming SSE responses are sealed end-to-end between your workstation (`127.0.0.1:18443`) and GPU memory.
- **v1.3.0 Hardening**:
  - Defaults to loopback `127.0.0.1` listening (opt-in `0.0.0.0` only when required by an external pod reverse proxy) with optional `verified_engine` SHA-256 binary integrity checks.
  - Selective IPython history scrubbing (erases only `KISSSHOT_`-related sensitive entries while preserving unrelated notebook work).
  - Optional `verify_runtime_lock(strict=False)` runtime lock verification (unlocked compatibility pass-through by default).
  - Frozen `kissshot-*` domain-separated AADs, 24h `KISSSHOT_TICKET` TTL enforcement, random dynamic routing (`/v1/ks-<hex>`), bounded request-ID replay protection, and upstream path SSRF allowlisting.

In [ ]:
# === Cell 1: Dependencies, Selective History Scrubbing, Runtime Lock & Passphrase Unlock ===
import subprocess, sys, os, json, hashlib, hmac, sqlite3, time, secrets, getpass
import importlib.metadata as importlib_metadata
from collections import deque

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

_SENSITIVE_HISTORY_PATTERNS = (
    "%KISSSHOT_%",
    "%SESSION_PASSPHRASE%",
    "%_TASK_HEX%",
    "%_LLM_HEX%",
)
_SENSITIVE_MEM_TOKENS = ("KISSSHOT_", "SESSION_PASSPHRASE", "_TASK_HEX", "_LLM_HEX")

def _scrub_sqlite_history_file(db_path: str) -> int:
    """K12: Selectively delete only KISSSHOT_-related sensitive rows; never wipe unrelated user work."""
    if not db_path or not os.path.exists(db_path):
        return 0
    removed = 0
    con = sqlite3.connect(db_path)
    try:
        cols = {row[1] for row in con.execute("PRAGMA table_info(history)").fetchall()}
        clauses = []
        params = []
        for pat in _SENSITIVE_HISTORY_PATTERNS:
            if "source" in cols:
                clauses.append("source LIKE ?")
                params.append(pat)
            if "source_raw" in cols:
                clauses.append("source_raw LIKE ?")
                params.append(pat)
        if clauses:
            cur = con.execute(f"DELETE FROM history WHERE {' OR '.join(clauses)}", params)
            removed += max(0, int(cur.rowcount or 0))
        tables = {row[0] for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
        if "output_history" in tables:
            out_cols = {row[1] for row in con.execute("PRAGMA table_info(output_history)").fetchall()}
            if "output" in out_cols:
                o_clauses = ["output LIKE ?" for _ in _SENSITIVE_HISTORY_PATTERNS]
                con.execute(f"DELETE FROM output_history WHERE {' OR '.join(o_clauses)}", list(_SENSITIVE_HISTORY_PATTERNS))
        con.commit()
        if removed > 0:
            con.execute("VACUUM")
    finally:
        con.close()
    return removed

def scrub_ipython_history(hist_db_path: str | None = None) -> int:
    """K12: Redact in-memory KISSSHOT_ entries and delete only KISSSHOT_ rows from IPython history SQLite."""
    removed = 0
    if hist_db_path is not None:
        return _scrub_sqlite_history_file(hist_db_path)
    try:
        ip = get_ipython()  # noqa: F821
        if ip and getattr(ip, "history_manager", None):
            hm = ip.history_manager
            hm.enabled = False
            if getattr(hm, "db", None):
                try:
                    cols = {row[1] for row in hm.db.execute("PRAGMA table_info(history)").fetchall()}
                    clauses, params = [], []
                    for pat in _SENSITIVE_HISTORY_PATTERNS:
                        if "source" in cols:
                            clauses.append("source LIKE ?")
                            params.append(pat)
                        if "source_raw" in cols:
                            clauses.append("source_raw LIKE ?")
                            params.append(pat)
                    if clauses:
                        cur = hm.db.execute(f"DELETE FROM history WHERE {' OR '.join(clauses)}", params)
                        removed += max(0, int(cur.rowcount or 0))
                        hm.db.commit()
                except Exception:
                    pass
        if ip and hasattr(ip, "user_ns"):
            ns = ip.user_ns
            for k in ("In", "_ih"):
                arr = ns.get(k)
                if isinstance(arr, list):
                    for idx in range(len(arr)):
                        text = str(arr[idx])
                        if any(tok in text for tok in _SENSITIVE_MEM_TOKENS):
                            arr[idx] = "# [cleared]"
            for out_k in ("Out", "_oh"):
                od = ns.get(out_k)
                if isinstance(od, dict):
                    for k in list(od.keys()):
                        text = str(od[k])
                        if any(tok in text for tok in _SENSITIVE_MEM_TOKENS):
                            od.pop(k, None)
    except Exception:
        pass
    hist_db = os.path.expanduser("~/.ipython/profile_default/history.sqlite")
    try:
        removed += _scrub_sqlite_history_file(hist_db)
    except Exception:
        pass
    return removed

scrub_ipython_history()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "cryptography", "huggingface_hub"])

RUNTIME_LOCK_PATH = os.environ.get("KISSSHOT_RUNTIME_LOCK", "").strip()
RUNTIME_LOCK_STRICT = False

try:
    from kiss_shot.runtime_lock import verify_runtime_lock  # type: ignore
except ImportError:
    _RUNTIME_LOCK_SCHEMA = "kissshot-runtime-lock-v1"

    def _probe_pkg_ver(pkg_name: str) -> str | None:
        for cand in (pkg_name, pkg_name.replace("_", "-"), pkg_name.replace("-", "_")):
            try:
                return importlib_metadata.version(cand)
            except importlib_metadata.PackageNotFoundError:
                continue
        return None

    def verify_runtime_lock(
        manifest: dict | None = None,
        *,
        profile: str | None = None,
        strict: bool = False,
        installed_versions: dict | None = None,
    ) -> dict:
        if not manifest:
            if strict:
                raise RuntimeError("Missing runtime lock manifest while strict=True")
            return {"ok": True, "allowed": True, "mode": "unlocked", "warnings": [], "missing": [], "mismatches": []}
        warnings, missing, mismatches = [], [], []
        schema = str(manifest.get("schema", ""))
        if schema != _RUNTIME_LOCK_SCHEMA:
            warnings.append(f"Unexpected schema: {schema!r}")
        m_profile = str(manifest.get("profile", ""))
        if profile and m_profile and m_profile != profile:
            warnings.append(f"Profile mismatch: expected {profile!r}, got {m_profile!r}")
        packages = manifest.get("packages")
        if not isinstance(packages, dict):
            raise ValueError("Runtime lock manifest 'packages' must be a dictionary")
        declared = str(manifest.get("manifest_sha256", "")).strip().lower()
        if declared:
            canon = json.dumps(
                {"schema": schema, "profile": m_profile, "packages": {k: packages[k] for k in sorted(packages)}},
                ensure_ascii=False, sort_keys=True, separators=(",", ":"),
            ).encode("utf-8")
            if not hmac.compare_digest(declared, hashlib.sha256(canon).hexdigest()):
                warnings.append("Runtime lock manifest_sha256 integrity check failed")
        for pkg_name, entry in packages.items():
            if not isinstance(entry, dict) or not entry.get("installed", True):
                continue
            locked_ver = str(entry.get("version", "")).strip()
            if not locked_ver:
                continue
            actual_ver = installed_versions[pkg_name] if (installed_versions is not None and pkg_name in installed_versions) else _probe_pkg_ver(pkg_name)
            if not actual_ver:
                missing.append(pkg_name)
            elif actual_ver != locked_ver:
                mismatches.append({"package": pkg_name, "expected": locked_ver, "actual": actual_ver})
        clean = not warnings and not missing and not mismatches
        if not clean and strict:
            raise RuntimeError(f"Runtime lock verification failed (warnings={warnings}, missing={missing}, mismatches={mismatches})")
        return {"ok": clean, "allowed": True if (clean or not strict) else False, "mode": "verified" if clean else "warn", "warnings": warnings, "missing": missing, "mismatches": mismatches}

_lock_manifest = json.loads(open(RUNTIME_LOCK_PATH, encoding="utf-8").read()) if (RUNTIME_LOCK_PATH and os.path.isfile(RUNTIME_LOCK_PATH)) else None
_lock_result = verify_runtime_lock(_lock_manifest, profile="llm", strict=RUNTIME_LOCK_STRICT)
print(f"🔒 Runtime lock check: mode={_lock_result['mode']} (ok={_lock_result['ok']})")

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

KDF_ITERATIONS = 210_000
SALT_BYTES, NONCE_BYTES, TAG_BYTES = 16, 12, 16
AAD_PAYLOAD = b"kissshot-payload-v2"
AAD_TICKET = b"kissshot-ticket-v2"
MAX_TTL_SECONDS = 86_400

def derive_key(pwd: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=KDF_ITERATIONS)
    return kdf.derive(pwd.encode("utf-8"))

def encrypt_payload_hex(data: dict, pwd: str, aad: bytes = AAD_PAYLOAD) -> str:
    salt, nonce = secrets.token_bytes(SALT_BYTES), secrets.token_bytes(NONCE_BYTES)
    wrapped = {"_ks_v": 2, "_ks_iat": int(time.time()), "d": data}
    pt = json.dumps(wrapped, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ct = AESGCM(derive_key(pwd, salt)).encrypt(nonce, pt, aad)
    return (salt + nonce + ct).hex()

def decrypt_payload_hex(hex_str: str, pwd: str, aad: bytes = AAD_PAYLOAD, max_age_sec: int = MAX_TTL_SECONDS) -> dict:
    raw = bytes.fromhex("".join(hex_str.split()))
    if len(raw) <= SALT_BYTES + NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated ciphertext")
    pt = AESGCM(derive_key(pwd, raw[:SALT_BYTES])).decrypt(
        raw[SALT_BYTES:SALT_BYTES + NONCE_BYTES],
        raw[SALT_BYTES + NONCE_BYTES:],
        aad,
    )
    parsed = json.loads(pt.decode("utf-8"))
    if isinstance(parsed, dict) and parsed.get("_ks_v") == 2 and "d" in parsed:
        iat = int(parsed.get("_ks_iat", 0) or 0)
        if not iat or abs(time.time() - iat) > max_age_sec:
            raise ValueError("Expired payload")
        return parsed["d"]
    return parsed

def aead_seal(cipher: AESGCM, pt: bytes, aad: bytes) -> bytes:
    nonce = secrets.token_bytes(NONCE_BYTES)
    return nonce + cipher.encrypt(nonce, pt, aad)

def aead_open(cipher: AESGCM, wire: bytes, aad: bytes) -> bytes:
    if len(wire) < NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated frame")
    return cipher.decrypt(wire[:NONCE_BYTES], wire[NONCE_BYTES:], aad)

SESSION_PASSPHRASE = getpass.getpass("🔑 Enter your Kiss-Shot passphrase (in-memory only): ").strip()
assert SESSION_PASSPHRASE, "Passphrase cannot be empty."
print("✅ Passphrase unlocked in memory (selective KISSSHOT_ history scrubbing active).")

In [ ]:
# === Cell 2: Start Hardened AEAD Gateway in Front of Local Inference Engine (127.0.0.1:8080) ===
# Set INGRESS_HTTPS_URL to your GPU pod's HTTPS proxy URL (e.g. https://<pod>-8081.proxy.runpod.net)
# or http://127.0.0.1:8081 when using SSH local port forwarding (-L 8081:127.0.0.1:8081).
INGRESS_HTTPS_URL = "http://127.0.0.1:8081"
UPSTREAM_ENGINE_URL = "http://127.0.0.1:8080"
# K13: Default strictly to loopback 127.0.0.1; only switch to "0.0.0.0" if your cloud pod ingress requires it.
GATEWAY_LISTEN_HOST = "127.0.0.1"
GATEWAY_LISTEN_PORT = 8081
# K13: Optional local engine binary SHA-256 verification before serving
ENGINE_BINARY_PATH = ""
ENGINE_BINARY_SHA256 = ""

from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import threading, urllib.request, urllib.error

def verified_engine(binary_path: str, expected_sha256: str = "", *, strict: bool = False) -> dict:
    """K13: Compute and verify SHA-256 integrity of the local inference engine binary before exposure."""
    real_path = os.path.realpath(str(binary_path or "").strip())
    if not real_path or not os.path.isfile(real_path):
        raise FileNotFoundError(f"Inference engine binary not found: {binary_path!r}")
    h = hashlib.sha256()
    with open(real_path, "rb") as bf:
        while True:
            chunk = bf.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    actual = h.hexdigest().lower()
    exp = str(expected_sha256 or "").strip().lower()
    if exp:
        if len(exp) != 64 or any(c not in "0123456789abcdef" for c in exp):
            raise ValueError(f"Invalid expected SHA-256 hex digest: {expected_sha256!r}")
        if not hmac.compare_digest(actual, exp):
            raise RuntimeError(
                f"Engine binary SHA-256 verification failed for {real_path}: expected {exp[:16]}..., got {actual[:16]}..."
            )
        return {"ok": True, "verified": True, "sha256": actual, "path": real_path}
    if strict:
        raise RuntimeError(f"Missing expected SHA-256 digest for {real_path} while strict=True")
    return {"ok": True, "verified": False, "sha256": actual, "path": real_path}

if ENGINE_BINARY_PATH or ENGINE_BINARY_SHA256:
    _eng_check = verified_engine(ENGINE_BINARY_PATH, ENGINE_BINARY_SHA256, strict=bool(ENGINE_BINARY_SHA256))
    print(f"🛡️ Verified engine binary: {_eng_check['path']} (sha256={_eng_check['sha256'][:16]}..., verified={_eng_check['verified']})")

ROUTE_PATH = f"/v1/ks-{secrets.token_hex(8)}"
SESSION_SALT = secrets.token_bytes(SALT_BYTES)
SESSION_CIPHER = AESGCM(derive_key(SESSION_PASSPHRASE, SESSION_SALT))
CONCURRENCY_SEM = threading.BoundedSemaphore(16)
ALLOWED_UPSTREAM_PATHS = {
    "/health",
    "/models",
    "/v1/models",
    "/v1/chat/completions",
    "/v1/completions",
    "/v1/embeddings",
    "/tokenize",
    "/detokenize",
}
_SEEN_IDS, _ID_QUEUE, _LOCK = set(), deque(), threading.Lock()

def _check_replay(req_id: str) -> bool:
    if not req_id or len(req_id) < 8 or len(req_id) > 64:
        return False
    with _LOCK:
        if req_id in _SEEN_IDS:
            return False
        if len(_ID_QUEUE) >= 4096:
            _SEEN_IDS.discard(_ID_QUEUE.popleft())
        _ID_QUEUE.append(req_id)
        _SEEN_IDS.add(req_id)
        return True

class NotebookAEADGateway(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    timeout = 15

    def log_message(self, fmt, *args):
        return

    def _deny(self, status: int = 403):
        raw = b'{"error":"forbidden"}'
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(raw)))
        self.send_header("Connection", "close")
        self.end_headers()
        self.wfile.write(raw)

    def do_GET(self):
        self._deny()

    def do_HEAD(self):
        self._deny()

    def do_POST(self):
        if self.path != ROUTE_PATH:
            self._deny()
            return
        if not CONCURRENCY_SEM.acquire(timeout=3.0):
            self._deny(503)
            return
        try:
            try:
                n = int(self.headers.get("Content-Length", "0") or "0")
                if n <= 0 or n > 64 * 1024 * 1024:
                    self._deny()
                    return
                wire = self.rfile.read(n)
                plain = aead_open(SESSION_CIPHER, wire, f"kissshot-c2s-v1:{ROUTE_PATH}".encode("utf-8"))
                meta_b, req_body = plain.split(b"\n", 1)
                meta = json.loads(meta_b.decode("utf-8"))
                req_id = str(meta.get("id", ""))
                method = str(meta.get("method", "POST")).upper()
                clean_path = str(meta.get("path", "")).split("?", 1)[0].rstrip("/") or "/"
                if (
                    abs(time.time() - int(meta.get("ts", 0))) > 300
                    or not _check_replay(req_id)
                    or method not in ("GET", "POST")
                    or clean_path not in ALLOWED_UPSTREAM_PATHS
                ):
                    self._deny()
                    return
            except Exception:
                self._deny()
                return

            up_req = urllib.request.Request(
                f"{UPSTREAM_ENGINE_URL.rstrip('/')}{clean_path}",
                data=(req_body if method == "POST" else None),
                headers={"Content-Type": "application/json"} if req_body else {},
                method=method,
            )
            try:
                with urllib.request.urlopen(up_req, timeout=300) as resp:
                    ct = (resp.headers.get("Content-Type") or "").lower()
                    if "text/event-stream" in ct:
                        self.send_response(200)
                        self.send_header("Content-Type", "text/event-stream; charset=utf-8")
                        self.send_header("Cache-Control", "no-cache")
                        self.send_header("Connection", "close")
                        self.end_headers()
                        seq = 0
                        while True:
                            ln = resp.readline(4 * 1024 * 1024)
                            if not ln:
                                break
                            enc = aead_seal(SESSION_CIPHER, ln, f"kissshot-s2c:{req_id}:{seq}".encode("ascii")).hex()
                            seq += 1
                            self.wfile.write(f"data: {enc}\n\n".encode("ascii"))
                            self.wfile.flush()
                        eof = aead_seal(SESSION_CIPHER, b"", f"kissshot-eof:{req_id}:{seq}".encode("ascii")).hex()
                        self.wfile.write(f"data: EOF:{eof}\n\n".encode("ascii"))
                        self.wfile.flush()
                    else:
                        out_b = resp.read(32 * 1024 * 1024)
                        hdr_b = json.dumps({"status": resp.status, "ct": "application/json"}).encode("utf-8")
                        enc_all = aead_seal(SESSION_CIPHER, hdr_b + b"\n" + out_b, f"kissshot-s2c-body:{req_id}".encode("ascii"))
                        self.send_response(200)
                        self.send_header("Content-Type", "application/octet-stream")
                        self.send_header("Content-Length", str(len(enc_all)))
                        self.send_header("Connection", "close")
                        self.end_headers()
                        self.wfile.write(enc_all)
            except urllib.error.HTTPError as he:
                err_b = he.read(4 * 1024 * 1024)
                hdr_b = json.dumps({"status": he.code, "ct": "application/json"}).encode("utf-8")
                enc_err = aead_seal(SESSION_CIPHER, hdr_b + b"\n" + err_b, f"kissshot-s2c-body:{req_id}".encode("ascii"))
                self.send_response(200)
                self.send_header("Content-Type", "application/octet-stream")
                self.send_header("Content-Length", str(len(enc_err)))
                self.send_header("Connection", "close")
                self.end_headers()
                self.wfile.write(enc_err)
        finally:
            CONCURRENCY_SEM.release()

ticket_payload = {
    "v": 2,
    "type": "kissshot_handshake",
    "url": INGRESS_HTTPS_URL.strip().rstrip("/"),
    "route": ROUTE_PATH,
    "salt": SESSION_SALT.hex(),
    "ts": int(time.time()),
}
ticket_line = "KISSSHOT_TICKET:" + encrypt_payload_hex(ticket_payload, SESSION_PASSPHRASE, aad=AAD_TICKET)

gw = ThreadingHTTPServer((GATEWAY_LISTEN_HOST, GATEWAY_LISTEN_PORT), NotebookAEADGateway)
print("=" * 76)
print(f"🎯 Remote AEAD Gateway Ready on {GATEWAY_LISTEN_HOST}:{GATEWAY_LISTEN_PORT}! Copy KISSSHOT_TICKET below:\n")
print(ticket_line)
print("\n" + "=" * 76)
try:
    gw.serve_forever()
except KeyboardInterrupt:
    gw.shutdown()